# ₿ Cryptocurrency Advanced Analysis
**Live data · CoinGecko API** | OHLC, volatility, correlation

---

In [ ]:
import pandas as pd, numpy as np, requests, time
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings; warnings.filterwarnings('ignore')
print("Libraries loaded")

## 1. Live Market Prices

In [ ]:
coins = "bitcoin,ethereum,dogecoin,solana,binancecoin,ripple,cardano"
url = (f"https://api.coingecko.com/api/v3/coins/markets?vs_currency=usd&ids={coins}"
       "&order=market_cap_desc&per_page=20&page=1&sparkline=false"
       "&price_change_percentage=24h,7d,30d")
prices = pd.json_normalize(requests.get(url, timeout=20).json())
prices[["name","current_price","market_cap","total_volume",
        "price_change_percentage_24h","ath"]].rename(columns={
    "name":"Coin","current_price":"Price ($)","market_cap":"Market Cap",
    "total_volume":"24h Volume","price_change_percentage_24h":"24h %","ath":"ATH"})

## 2. Market Cap Comparison

In [ ]:
prices["Market Cap ($B)"] = prices["market_cap"] / 1e9
fig = px.bar(prices.sort_values("Market Cap ($B)", ascending=True),
             x="Market Cap ($B)", y="name", orientation="h",
             color="price_change_percentage_24h",
             color_continuous_scale="RdYlGn", color_continuous_midpoint=0,
             title="Market Cap ($B) — colored by 24h % Change")
fig.update_layout(height=450)
fig.show()

## 3. Bitcoin OHLC Candlestick (90 days)

In [ ]:
ohlc = requests.get(
    "https://api.coingecko.com/api/v3/coins/bitcoin/ohlc?vs_currency=usd&days=90",
    timeout=20).json()
btc = pd.DataFrame(ohlc, columns=["ts","Open","High","Low","Close"])
btc["Date"] = pd.to_datetime(btc["ts"], unit="ms")
fig = go.Figure(go.Candlestick(
    x=btc["Date"], open=btc["Open"], high=btc["High"],
    low=btc["Low"], close=btc["Close"],
    increasing_line_color="#26a69a", decreasing_line_color="#ef5350"))
fig.update_layout(title="Bitcoin — 90-Day OHLC Candlestick",
                  xaxis_rangeslider_visible=False, height=500)
fig.show()

## 4. Price + Volume Chart

In [ ]:
mc = requests.get(
    "https://api.coingecko.com/api/v3/coins/bitcoin/market_chart?vs_currency=usd&days=90",
    timeout=20).json()
btc_pv = pd.DataFrame(mc["prices"], columns=["ts","Price"])
btc_pv["Volume"] = [v[1] for v in mc["total_volumes"]]
btc_pv["Date"] = pd.to_datetime(btc_pv["ts"], unit="ms")

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.7,0.3],
                    vertical_spacing=0.05)
fig.add_trace(go.Scatter(x=btc_pv["Date"], y=btc_pv["Price"],
    name="Price", line=dict(color="orange")), row=1, col=1)
fig.add_trace(go.Bar(x=btc_pv["Date"], y=btc_pv["Volume"],
    name="Volume", marker_color="steelblue", opacity=0.5), row=2, col=1)
fig.update_layout(title="Bitcoin Price & Volume (90d)",
                  hovermode="x unified", height=520)
fig.show()

## 5. Multi-Coin Normalized Performance

In [ ]:
coin_ids = {"Bitcoin":"bitcoin","Ethereum":"ethereum","Solana":"solana","Dogecoin":"dogecoin"}
all_dfs = []
for name, cid in coin_ids.items():
    try:
        d = requests.get(
            f"https://api.coingecko.com/api/v3/coins/{cid}/market_chart"
            "?vs_currency=usd&days=90", timeout=20).json()
        tmp = pd.DataFrame(d["prices"], columns=["ts","Price"])
        tmp["Date"] = pd.to_datetime(tmp["ts"], unit="ms")
        tmp["Coin"] = name
        tmp["Normalized"] = tmp["Price"] / tmp["Price"].iloc[0] * 100
        all_dfs.append(tmp)
        time.sleep(1)  # respect rate limit
    except Exception as e:
        print(f"Skipped {name}: {e}")
comp = pd.concat(all_dfs, ignore_index=True)
fig = px.line(comp, x="Date", y="Normalized", color="Coin",
              title="Normalized Performance — 90 Days (base = 100)")
fig.update_layout(hovermode="x unified", height=450)
fig.show()

## 6. Volatility Analysis

In [ ]:
pivot = comp.pivot(index="Date", columns="Coin", values="Price").dropna()
returns = pivot.pct_change().dropna()
print("Daily Return Statistics:")
print(returns.describe().round(5))

vol = (returns.std() * 100).reset_index()
vol.columns = ["Coin","Daily Volatility (%)"]
fig = px.bar(vol, x="Coin", y="Daily Volatility (%)",
             color="Daily Volatility (%)", color_continuous_scale="Oranges",
             title="Daily Return Volatility by Coin")
fig.update_layout(height=380)
fig.show()

## 7. Return Correlation Matrix

In [ ]:
fig = px.imshow(returns.corr(), color_continuous_scale="RdBu_r",
                zmin=-1, zmax=1, text_auto=".2f",
                title="Coin Return Correlation Matrix", aspect="auto")
fig.update_layout(height=420)
fig.show()

## 8. Rolling 7-Day Volatility

In [ ]:
roll_vol = returns.rolling(7).std() * 100
fig = px.line(roll_vol.reset_index().melt("Date", var_name="Coin", value_name="7d Rolling Vol (%)"),
              x="Date", y="7d Rolling Vol (%)", color="Coin",
              title="7-Day Rolling Volatility")
fig.update_layout(hovermode="x unified", height=420)
fig.show()

## 9. Key Insights

In [ ]:
print("=== CRYPTO KEY INSIGHTS ===")
for _, row in prices.iterrows():
    chg = row.get("price_change_percentage_24h") or 0
    print(f"{row['name']:15s}  Price: ${row['current_price']:>12,.4f}  24h: {chg:+.2f}%")
most_vol = vol.nlargest(1,'Daily Volatility (%)')['Coin'].values[0]
print(f"\nMost volatile (90d): {most_vol}")
best = comp.groupby("Coin").apply(
    lambda x: (x['Price'].iloc[-1]/x['Price'].iloc[0]-1)*100).idxmax()
print(f"Best 90d performer : {best}")